# 1 — White-box attacks on a pretrained ResNet-20

Four L-infinity attacks against `chenyaofo/pytorch-cifar-models`'
`cifar10_resnet20`, which scores **92.59%** on the clean CIFAR-10 test set:

| attack | what it adds |
|---|---|
| FGSM | one signed-gradient step |
| PGD | many small steps, projected back into the epsilon ball each time |
| MI-FGSM | PGD plus a momentum term on the L1-normalised gradient |
| APGD | PGD with an adaptive step size, momentum and restarts from the best point |

All four get the same budget, so the numbers are directly comparable.

`advlab` defines every attack on images in `[0, 1]`, so `eps=8/255` is eight
levels of an 8-bit pixel — the standard CIFAR-10 L-infinity budget — and it
means the same thing in all four notebooks.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import advlab
from advlab import CLASSES, DEVICE, accuracy, apgd, cifar10, fgsm, mifgsm, pgd

advlab.set_seed()
print("device:", DEVICE)

In [ ]:
EPS = 8 / 255
test_loader = cifar10(train=False, batch_size=256)

model = advlab.resnet20(pretrained=True)
clean = accuracy(model, test_loader)
print(f"clean test accuracy: {clean:.2f}%")

## Which images does the pretrained model already get wrong?

Before attacking anything, it is worth seeing the baseline's own errors — they
are the confusions the attacks will be pushing on.

In [ ]:
model.eval()
wrong = []
for x, y in test_loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    with torch.no_grad():
        pred = model(x).argmax(1)
    for i in (pred != y).nonzero(as_tuple=True)[0].tolist():
        wrong.append((x[i].cpu(), y[i].item(), pred[i].item()))
    if len(wrong) >= 9:
        break

fig, axes = plt.subplots(3, 3, figsize=(6, 6.4))
for ax, (img, true, pred) in zip(axes.ravel(), wrong[:9]):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"{CLASSES[true]} -> {CLASSES[pred]}", fontsize=9)
    ax.axis("off")
fig.suptitle("First 9 misclassified test images")
fig.tight_layout()
plt.show()

## FGSM

$$x_{adv} = \operatorname{clip}\big(x + \epsilon \cdot
\operatorname{sign}(\nabla_x J(\theta, x, y))\big)$$

One gradient evaluation. The sign means every pixel moves by the full
`eps`, so the perturbation sits on a corner of the epsilon ball.

In [ ]:
acc_fgsm = accuracy(model, test_loader, attack=fgsm, eps=EPS)
print(f"FGSM  eps={EPS:.4f} ({EPS * 255:.0f}/255): {acc_fgsm:.2f}%"
      f"   (drop {clean - acc_fgsm:.2f})")

x, y = next(iter(test_loader))
advlab.show_attack_grid(model, x, y, fgsm, f"FGSM, eps={EPS * 255:.0f}/255",
                        eps=EPS)
plt.show()

## PGD

$$x^{t+1} = \Pi_{B_\epsilon(x)}\big(x^t + \alpha \cdot
\operatorname{sign}(\nabla_x J(\theta, x^t, y))\big)$$

Ten steps of `alpha = 2/255` from a random point in the epsilon ball. Each
step is projected back into the ball and clamped to the valid image range, so
the total perturbation never exceeds `eps` however many steps are taken.

In [ ]:
acc_pgd = accuracy(model, test_loader, attack=pgd, eps=EPS,
                   alpha=2 / 255, steps=10)
print(f"PGD   eps={EPS:.4f}, alpha=2/255, 10 steps: {acc_pgd:.2f}%"
      f"   (drop {clean - acc_pgd:.2f})")

advlab.show_attack_grid(model, x, y, pgd, f"PGD, eps={EPS * 255:.0f}/255",
                        eps=EPS)
plt.show()

## MI-FGSM — PGD with momentum

$$g^{t+1} = \mu \cdot g^t +
\frac{\nabla_x J(\theta, x^t, y)}{\lVert \nabla_x J(\theta, x^t, y) \rVert_1},
\qquad
x^{t+1} = \Pi_{B_\epsilon(x)}\big(x^t + \alpha \cdot
\operatorname{sign}(g^{t+1})\big)$$

The L1 normalisation is the part that matters. Gradient magnitudes vary by
orders of magnitude between steps and between samples; dividing by the L1 norm
puts every step on the same scale, so `mu` controls how much history carries
over. Accumulating `sign(grad)` instead would leave `mu` doing nothing but
decay.

In [ ]:
acc_mi = accuracy(model, test_loader, attack=mifgsm, eps=EPS,
                  alpha=2 / 255, steps=10, mu=1.0)
print(f"MI-FGSM  mu=1.0, 10 steps: {acc_mi:.2f}%"
      f"   (drop {clean - acc_mi:.2f})")

advlab.show_attack_grid(model, x, y, mifgsm,
                        f"MI-FGSM, eps={EPS * 255:.0f}/255", eps=EPS)
plt.show()

## APGD (Croce & Hein, 2020)

The step size is not a hyperparameter here. It starts at `2 * eps` and halves
at a fixed schedule of checkpoints whenever progress has stalled — fewer than
`rho = 0.75` of the steps since the last checkpoint improved the objective, or
neither the step size nor the best objective moved at all. On halving, the
search jumps back to the best point found so far instead of continuing from
wherever it had drifted to. Updates carry momentum with `alpha = 0.75`.

Two objectives, both maximised:

- **CE** — cross-entropy, the usual choice.
- **CW** — the Carlini-Wagner margin `max_{i != y} z_i - z_y`, which is
  positive exactly when the sample is already misclassified. It cares only
  about crossing the decision boundary, not about how confidently.

In [ ]:
acc_apgd_ce = accuracy(model, test_loader, attack=apgd, eps=EPS,
                       steps=20, loss="ce")
acc_apgd_cw = accuracy(model, test_loader, attack=apgd, eps=EPS,
                       steps=20, loss="cw")
print(f"checkpoints for 20 steps: {advlab._apgd_checkpoints(20)}")
print(f"APGD-CE  20 steps: {acc_apgd_ce:.2f}%   (drop {clean - acc_apgd_ce:.2f})")
print(f"APGD-CW  20 steps: {acc_apgd_cw:.2f}%   (drop {clean - acc_apgd_cw:.2f})")

In [ ]:
for name, loss in [("CE", "ce"), ("CW", "cw")]:
    advlab.show_attack_grid(
        model, x, y, apgd, f"APGD-{name}, eps={EPS * 255:.0f}/255",
        eps=EPS, steps=20, loss=loss,
    )
    plt.show()

## Comparison

In [ ]:
rows = [
    ("none (clean)", clean, "baseline, no perturbation"),
    ("FGSM", acc_fgsm, "one step; fast, weakest"),
    ("PGD", acc_pgd, "10 steps; much stronger"),
    ("MI-FGSM", acc_mi, "momentum stabilises the direction"),
    ("APGD-CE", acc_apgd_ce, "adaptive step size, no tuning"),
    ("APGD-CW", acc_apgd_cw, "margin objective, boundary-focused"),
]
print(f"{'attack':14s} {'accuracy':>9s}  {'drop':>6s}  note")
for name, acc, note in rows:
    print(f"{name:14s} {acc:8.2f}%  {clean - acc:6.2f}  {note}")

**Strengths and weaknesses.**

- **FGSM** — one gradient evaluation, so it is the cheapest way to get a first
  reading on a model. It is also the weakest: a single full-`eps` step lands on
  a ball corner chosen by the local gradient, which is a poor estimate of the
  worst case, and it is close to useless against a model with any real defence.
- **PGD** — the standard for a reason. Ten steps cost ten times as much and
  buy a much better inner maximisation. The cost is two hyperparameters,
  `alpha` and `steps`, and the result is sensitive to both.
- **MI-FGSM** — momentum smooths the trajectory and helps escape flat regions;
  it mainly pays off for transfer to other models, which is not tested here.
  Adds `mu` to tune.
- **APGD** — strongest and hyperparameter-free, which is what makes it the
  right tool for a final robustness number: a defence cannot look good merely
  because the evaluator picked a bad step size. It is the most complex to
  implement and the slowest per step.